In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
books = pd.read_csv('processed datasets/Books.csv')
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002-01-01,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001-01-01,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991-01-01,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999-01-01,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999-01-01,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [3]:
ratings = pd.read_csv('processed datasets/ratings.csv')


In [4]:
#there is error for loading images and also we wont need it as we dont have front end
books = books.iloc[: ,0:5]
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002-01-01,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001-01-01,HarperFlamingo Canada
2,0060973129,Decision in Normandy,Carlo D'Este,1991-01-01,HarperPerennial
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999-01-01,Farrar Straus Giroux
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999-01-01,W. W. Norton &amp; Company


In [5]:
#for now lets conver date to year only
books['Year-Of-Publication'] = pd.to_datetime(books['Year-Of-Publication']).dt.year
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company


In [6]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271351 entries, 0 to 271350
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271351 non-null  object
 1   Book-Title           271351 non-null  object
 2   Book-Author          271351 non-null  object
 3   Year-Of-Publication  271351 non-null  int32 
 4   Publisher            271351 non-null  object
dtypes: int32(1), object(4)
memory usage: 9.3+ MB


## ratings data


In [7]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1149780 entries, 0 to 1149779
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   User-ID      1149780 non-null  int64 
 1   ISBN         1149780 non-null  object
 2   Book-Rating  1149780 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 26.3+ MB


In [8]:
ratings['Book-Rating'].value_counts()

Book-Rating
0     716109
8     103736
10     78610
7      76457
9      67541
5      50974
6      36924
4       8904
3       5996
2       2759
1       1770
Name: count, dtype: int64

In [9]:
ratings = ratings[ratings['Book-Rating'] != 0]
ratings.info()

<class 'pandas.core.frame.DataFrame'>
Index: 433671 entries, 1 to 1149779
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   User-ID      433671 non-null  int64 
 1   ISBN         433671 non-null  object
 2   Book-Rating  433671 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 13.2+ MB


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [11]:
books['combined_features'] = books['Book-Title'] + ' ' + books['Book-Author']

In [12]:
tfidf = TfidfVectorizer(max_features= 8000 , stop_words='english' , lowercase= True)

tfidf_matrix = tfidf.fit_transform(books['combined_features'])

print(f'TF-IDF matrix shape : {tfidf_matrix.shape}')
print(f"Number of feature : {len(tfidf.get_feature_names_out())}")

TF-IDF matrix shape : (271351, 8000)
Number of feature : 8000


In [13]:
# <!-- we want ot crete uuser1 = matrix[movie1 ,move 2]... -->
# <!-- so we went connected the ratings df and idfmatrix using indexs values for comparasion  -->

### we did this only on first 100 books (my memory was less) ,change below code include all books

In [14]:
user_profiles = {}

#fetch user's liked move id and create list of its indexes in books df.This index is used further to fetch the tfidfmatrix 
print('Index for users and its rated books : ')


for user_id in ratings['User-ID'].unique()[:100]:       #chnage this to include all  books
    user_books = ratings[ratings['User-ID'] == user_id]['ISBN'].values

    #applied this condition becz of error of mismatch data(no of ratings peruser != no of books(books not found in df))
    for book in user_books:
        if book in books['ISBN'].unique():
            user_book_ratings = ratings[(ratings['User-ID'] == user_id) & (ratings['ISBN'] == book)]['Book-Rating'].values

    book_indices = books[books['ISBN'].isin(user_books)].index.tolist()

    print(user_id , book_indices)

    if book_indices:
        # Get vectors for user's books
        user_tfidf = tfidf_matrix[book_indices]
        # print(user_id ,user_tfidf.shape , user_book_ratings.shape)
        
        #the vector is multiplied bsy users rating so inc the imp of particular book..
        weighted_vectors = []
        for i, rating in enumerate(user_book_ratings):
        
            weighted_vector = user_tfidf[i] * rating
            weighted_vectors.append(weighted_vector)
        
        #sum all  vectors to create user profile
        user_profile = sum(weighted_vectors)
        user_profiles[user_id] = user_profile
        
        # print(f"User profile shape: {user_profile.shape}")
    


276726 [225809]
276729 [246830, 246831]
276736 []
276737 []
276744 [9294]
276745 []
276747 [1836, 4779, 6276, 246832, 246833]
276748 [120675]
276751 [37678]
276754 [1986]
276755 [102]
276760 []
276762 [4884, 7817]
276768 []
276772 [27220, 33827, 83625]
276774 [67303]
276780 []
276786 [117691, 246842]
276788 [2238, 5506, 19991]
276796 [166]
276798 [3217, 82225, 118264]
276800 [246847]
276804 [16780]
276808 [46586]
276811 [5731]
276812 []
276813 [28596, 76824, 95312, 164412, 246852]
276814 [14815, 83893]
276820 []
276822 [1027, 15601, 17083, 27515, 34235, 50709, 80668, 88353, 115318, 140328, 178108, 216307, 233858]
276827 []
276828 [33877]
276830 [13927]
276832 [1710]
276835 [246853]
276837 [4757]
276842 []
276847 [5041, 5213, 5732, 8271, 13612, 14438, 26788, 29865, 34538, 34539, 34540, 37702, 37723, 49230, 51260, 51921, 51923, 51924, 52797, 52798, 67463, 82880, 89546, 113952, 114609, 125294, 210910, 246857, 246858, 246860, 246861, 246862, 246863]
276848 []
276850 [11128]
276853 [132387,

In [15]:
def recommend_book_for_user(user_id, top_n=5):
    user_profile = user_profiles.get(user_id)
    if user_profile is None:
        return False
    
    
    already_rated_books = set(ratings[ratings['User-ID'] == user_id]['ISBN'].values)

    # It measures how each book is closer to user
    # bala ithee xi + yj + zk vector is compared with all other vectors [[...][...][...]]
    similarities = cosine_similarity(user_profile, tfidf_matrix).flatten()
    

    # Get indices of unrated books
    unrated_books_index = [i for i, isbn in enumerate(books['ISBN']) if isbn not in already_rated_books]

    #Filter out similaraties for unrated books only to avoid duble suggestions 
    #[index of similarities ] == [index of unrated books]
    unrated_books_similarities = [(i, similarities[i]) for i in unrated_books_index]

    #lamba ensure that sorting in set() is done basis on values and not index
    unrated_books_similarities.sort(key=lambda x: x[1], reverse=True)

    
    top_indices = [i for i, sim in unrated_books_similarities[:top_n]]
    recommended_books = books.iloc[top_indices]
    
    return recommended_books[['ISBN', 'Book-Title', 'Book-Author']]


### we see duplicates becz the publisher of each was diff .There is no duplicates in original books dataset


In [16]:
recommend_book_for_user(277051)

,ISBN,Book-Title,Book-Author
12250,0385501560,Choke,Chuck Palahniuk
64739,2207253635,Choke,Chuck Palahniuk
72644,2070756270,Survivant,Chuck Palahniuk
92728,3442541670,Flug 2039.,Chuck Palahniuk
254776,0802711502,Codename: Cipher,Chuck Freadhoff


In [17]:
# # Set numpy print options to show the full array
# np.set_printoptions(threshold=np.inf, linewidth=np.inf)
# print(user_profiles[user_id].toarray().flatten())